# Clean and store Claims data

In [1]:

# ===============================
# 1. LOAD RAW DATA (Bronze Layer)
# ===============================
df_customers_raw = spark.read.parquet('abfss://ws_insurance_dev@onelake.dfs.fabric.microsoft.com/lh_bronze.Lakehouse/Files/insurance_raw/customers.parquet')
df_policies_raw  = spark.read.parquet('abfss://ws_insurance_dev@onelake.dfs.fabric.microsoft.com/lh_bronze.Lakehouse/Files/insurance_raw/policies.parquet')
df_claims_raw    = spark.read.parquet('abfss://ws_insurance_dev@onelake.dfs.fabric.microsoft.com/lh_bronze.Lakehouse/Files/insurance_raw/claims.parquet')

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 5, Finished, Available, Finished, False)

## Great Expectation Function definitions for data validation

In [2]:
import great_expectations as gx
from pyspark.sql import DataFrame
from datetime import datetime 
from pyspark.sql.functions import lit, current_timestamp, col, year, month, dayofmonth

def validate_schema_with_gx(
    df: DataFrame,
    schema: dict,
    expected_row_count: int = None,
    check_ordered_columns: bool = True,
    enable_length_check: bool = False
) -> bool:
    ''' Function to validates input dataframe against a schema using Great Expectations.
    Input: df (dataframe), schema (dict), expected_row_count (int), check_ordered_columns (bool), enable_length_check (bool)
    Output: bool
    '''
   
    """
    Runs Great Expectations checks on a Spark DataFrame.
    """

    
    # 1) Build a transient GX context and Spark datasource
    context = gx.get_context()
    ds = context.data_sources.add_spark(name="spark_in_memory")
    asset = ds.add_dataframe_asset(name="df_asset")
    batch_def = asset.add_batch_definition_whole_dataframe("df_batch")
    batch = batch_def.get_batch(batch_parameters={"dataframe": df})
    """
    # 1) Build a transient GX context and Pandas Dataframe datasource
    context = gx.get_context()
    datasource = context.data_sources.add_pandas(name="pandas_datasource")
    name = "df_dataframe"
    data_asset = datasource.add_dataframe_asset(name=name)
    batch_definition_name = "df_batch"
    batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
    )
    batch_parameters = {"dataframe": df}
    
    # Get the dataframe as a Batch
    batch = batch_definition.get_batch(batch_parameters=batch_parameters)
    """
    # 2) Run expectations per schema
    from great_expectations import expectations as E
    results = []
    ordered_cols = []
    for col, props in schema.items():
        ordered_cols.append(col)

        if props.get("unique", False):
            results.append(batch.validate(E.ExpectColumnValuesToBeUnique(column=col)))
        if props.get("nullable", True) is False:
            results.append(batch.validate(E.ExpectColumnValuesToNotBeNull(column=col)))

        dtype = props.get("dtype")
        if dtype:
            results.append(batch.validate(E.ExpectColumnValuesToBeOfType(column=col, type_=dtype)))

        if enable_length_check:
            size = props.get("size")
            if size is not None:
                results.append(
                    batch.validate(
                        E.ExpectColumnValueLengthsToBeBetween(
                            column=col, min_value=None, max_value=int(size), strict_max=True
                        )
                    )
                )
    
    # 3) Table-level expectations
    if check_ordered_columns:
        results.append(batch.validate(E.ExpectTableColumnsToMatchOrderedList(column_list=ordered_cols)))
    if expected_row_count is not None:
        results.append(batch.validate(E.ExpectTableRowCountToEqual(value=int(expected_row_count))))

    # 4) Summarize results
    temp = list()
    total = len(results)
    for r in results:
        if getattr(r, "success", False):
            temp.append(1)
        
    successes = len(temp)
    #successes = sum(1 for r in results if getattr(r, "success", False))
    failures = total - successes
    
    #total = len(results)
    #successes = sum(1 for r in results if getattr(r, "success", False))
    #failures = total - successes

    print(f"[DQ] Expectations run: {total} | Passed: {successes} | Failed: {failures}")
    if failures > 0:
        for r in results:
            if not getattr(r, "success", False):
                cfg = getattr(r, "expectation_config", None)
                etype = getattr(cfg, "type", "unknown") if cfg else "unknown"
                kwargs = getattr(cfg, "kwargs", {}) if cfg else {}
                print(f"[DQ][FAIL] {etype} {kwargs}")
        print("[DQ] Data Quality validation failed.")
        return False
        #raise Exception("Data Quality validation failed.")
    else:
        print("[DQ] All checks passed ✔️")
        return True



StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 6, Finished, Available, Finished, False)

## Create schema definition to perform validation test with great expectations

In [3]:
# Expect the columns to be from the expected column set
spark_customers_expected_schema = {
    "cust_id":         {"size": 2, "dtype": "StringType",  "unique": True,  "nullable": True},
    "fullname":       {"size": 255,  "dtype": "StringType",   "unique": False, "nullable": True},
    "date_of_birth":       {"size": 50,  "dtype": "StringType",   "unique": False, "nullable": True},
    "Gender":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    "contact_number":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    
}

spark_claims_expected_schema = {
    "claim_id":         {"size": 3, "dtype": "StringType",  "unique": True,  "nullable": True},
    "policy_id":       {"size": 2,  "dtype": "StringType",   "unique": False, "nullable": True},
    "claim_date":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    "claim_amount":       {"size": 10,  "dtype": "StringType",   "unique": False, "nullable": True},
    "status":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    "Gender":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    "deductible":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    "claim_reason":       {"size": 50,  "dtype": "StringType",   "unique": False, "nullable": True},
    "is_fraud":       {"size": 5,  "dtype": "StringType",   "unique": False, "nullable": True},
    "Zip_Code":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
}

spark_policies_expected_schema = {
    "policy_id":         {"size": 2, "dtype": "StringType",  "unique": True,  "nullable": True},
    "cust_id":       {"size": 2,  "dtype": "StringType",   "unique": False, "nullable": True},
    "policy_type":       {"size": 50,  "dtype": "StringType",   "unique": False, "nullable": True},
    "status":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    "start_date":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    "end_date":       {"size": 25,  "dtype": "StringType",   "unique": False, "nullable": True},
    "coverage_amount":       {"size": 10,  "dtype": "StringType",   "unique": False, "nullable": True},
    "annual_premium":       {"size": 10,  "dtype": "StringType",   "unique": False, "nullable": True},
    "monthly_payment":       {"size": 10,  "dtype": "StringType",   "unique": False, "nullable": True},
    "distribution_channel":       {"size": 10,  "dtype": "StringType",   "unique": False, "nullable": True},
}




StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 7, Finished, Available, Finished, False)

## Get dataset row counts

In [4]:
customers_expected_rows = df_customers_raw.count()
policies_expected_rows = df_policies_raw.count()
claims_expected_rows = df_claims_raw.count()

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 8, Finished, Available, Finished, False)

## Applied validation test to data. Rules include:
### 1) Schema validation
### 2) Records count validation
### 3) Column order validation

In [5]:
# TO USE WITH A COMPUTE CONFIGURATION
customers_validated = validate_schema_with_gx(
    df=df_customers_raw,
    schema=spark_customers_expected_schema,
    expected_row_count=customers_expected_rows,
    check_ordered_columns=True,
    enable_length_check=False
)


policies_validated = validate_schema_with_gx(
    df=df_policies_raw,
    schema=spark_policies_expected_schema,
    expected_row_count=policies_expected_rows,
    check_ordered_columns=True,
    enable_length_check=False
)

claims_validated = validate_schema_with_gx(
    df=df_claims_raw,
    schema=spark_claims_expected_schema,
    expected_row_count=claims_expected_rows,
    check_ordered_columns=True,
    enable_length_check=False
)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 9, Finished, Available, Finished, False)

Calculating Metrics:   0%|          | 0/12 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

...

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 11, Finished, Available, Finished, False)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 12, Finished, Available, Finished, False)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 13, Finished, Available, Finished, False)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 14, Finished, Available, Finished, False)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 15, Finished, Available, Finished, False)

# Save to delta table if schema validation test passes

In [9]:
if customers_validated:
    df_customers_raw.write.mode("overwrite").format("delta").partitionBy("Gender").option("mergeSchema", "true").saveAsTable("bronze.customers")
else:
    raise Exception("Bronze customers table schema validation failed")

if policies_validated:
    df_policies_raw.write.mode("overwrite").format("delta").partitionBy("policy_type").option("mergeSchema", "true").saveAsTable("bronze.policies")
else:
    raise Exception("Bronze policies table schema validation failed")

if claims_validated:
    df_claims_raw.write.mode("overwrite").format("delta").partitionBy("Zip_Code").option("mergeSchema", "true").saveAsTable("bronze.claims")
else:
    raise Exception("Bronze claims table schema validation failed")

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 18, Finished, Available, Finished, False)

### Cleaning the customer data

In [10]:
from pyspark.sql.functions import *

df_customers_clean = (
    df_customers_raw
    .dropna(subset=["cust_id"])
    .withColumn("cust_id", col("cust_id").cast("int"))
    .withColumn("CustomerName", initcap(trim(col("fullname"))))
    .withColumn("Date_of_Birth", to_date(col("date_of_birth")))
    .withColumn("Contact_Number", when(col("contact_number").isNull(), "Not Provided").otherwise(col("contact_number")))
    .withColumn("Gender", when(col("Gender").isNull(), "Not Provided").otherwise(col("Gender")))
    .drop("fullname")
    .dropDuplicates()
)

display(df_customers_clean)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 705c0b9e-e415-4347-9074-9ab0e7739807)

### Cleaning the policy data

In [ ]:
display(df_policies_raw)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, -1, Cancelled, , Cancelled, True)

In [11]:
df_policies_clean = (
    df_policies_raw
    .withColumn("policy_type", initcap(trim(col("policy_Type"))))
    .withColumn("status", initcap(trim(col("status"))))
    # Standardize date format using multiple patterns
    .withColumn("start_date", to_date(
        coalesce(
            to_date(col("start_date"), "MM/dd/yyyy"),
            to_date(col("start_date"), "yyyy/MM/dd"),
            to_date(col("start_date"), "yyyy-MM-dd"),
            to_date(col("start_date"), "dd-MM-yyyy"),
            to_date(col("start_date"), "dd-MMM-yyyy"),
            to_date(col("start_date"), "MM-dd-yyyy"),
            to_date(col("start_date"), "yyyy.MM.dd"),
            to_date(col("start_date"), "dd/MM/yyyy"),
            to_date(col("start_date"), "dd.MM.yyyy"),
            to_date(col("start_date"), "MMMM dd yyyy")
        )
    ))
    .withColumn("end_date", to_date(
        coalesce(
            to_date(col("end_date"), "MM/dd/yyyy"),
            to_date(col("end_date"), "yyyy/MM/dd"),
            to_date(col("end_date"), "yyyy-MM-dd"),
            to_date(col("end_date"), "dd-MM-yyyy"),
            to_date(col("end_date"), "dd-MMM-yyyy"),
            to_date(col("end_date"), "MM-dd-yyyy"),
            to_date(col("end_date"), "yyyy.MM.dd"),
            to_date(col("end_date"), "dd/MM/yyyy"),
            to_date(col("end_date"), "dd.MM.yyyy"),
            to_date(col("end_date"), "MMMM dd yyyy")
        )
    ))
    .withColumn("coverage_amount", when(col("Coverage_Amount") == "not available", None)
                .otherwise(col("Coverage_Amount").cast("double")))
    .withColumn("annual_premium", when(col("annual_premium") == "not available", None)
                .otherwise(col("annual_premium").cast("double")))
    .withColumn("monthly_payment", when(col("monthly_payment") == "not available", None)
                .otherwise(col("monthly_payment").cast("double")))
    .withColumn("distribution_channel", initcap(trim(col("distribution_channel"))))
    .withColumn("cust_id", col("cust_id").cast("int"))
    .withColumn("policy_year", year(to_date(col("end_date"), "MMMM dd yyyy")))
    .withColumn("policy_month", month(to_date(col("end_date"), "MMMM dd yyyy")))
    .withColumn("policy_day", dayofmonth(to_date(col("end_date"), "MMMM dd yyyy")))
    .dropna(subset=["cust_id", "policy_id"])
    .dropDuplicates()
)
display(df_policies_clean)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 15e7823d-8bd8-48cd-af5e-70806c755cbd)

### Cleaning the Claim data

In [ ]:
display(df_claims_raw)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, -1, Cancelled, , Cancelled, True)

In [12]:
df_claims_clean = (
    df_claims_raw
    .withColumn("claim_date", to_date(col("claim_date")))
    .withColumn("claim_amount", col("claim_amount").cast("double"))
    .withColumn("status", initcap(trim(col("status"))))
    .withColumn("gender", initcap(trim(col("Gender"))))
    .withColumn("deductible", when(col("deductible") == "not available", None)
                .otherwise(col("deductible").cast("double")))
    .withColumn("claims_year", year(to_date(col("claim_date"))))
    .withColumn("claims_month", month(to_date(col("claim_date"))))
    .withColumn("claims_day", dayofmonth(to_date(col("claim_date"))))
    .dropna(subset=["claim_id"])
    .dropDuplicates()
)
display(df_claims_clean)

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b471caf-85f6-43b5-9b09-1ee6220e5aeb)

#### Create delta table for clean datasets

In [13]:
spark.sql("DROP TABLE IF EXISTS bronze.clean_customers")
spark.sql("DROP TABLE IF EXISTS bronze.clean_policies")
spark.sql("DROP TABLE IF EXISTS bronze.clean_claims")


df_customers_clean.write.mode("overwrite").format("delta").option("mergeSchema", "true").partitionBy("gender").saveAsTable("bronze.clean_customers")
df_policies_clean.write.mode("overwrite").format("delta").option("mergeSchema", "true").partitionBy("policy_year", "policy_month", "policy_day").saveAsTable("bronze.clean_policies")
df_claims_clean.write.mode("overwrite").format("delta").option("mergeSchema", "true").partitionBy("claims_year", "claims_month", "claims_day").saveAsTable("bronze.clean_claims")

StatementMeta(, a83623d0-323c-4a0a-b0b6-8983f31d3ea9, 22, Finished, Available, Finished, False)